# 🔒 Phishing Detector - LSTM + Attention Training on GPU

This notebook trains the phishing detection model using Google Colab's T4 GPU.

**Steps:**
1. Upload your `PhiUSIIL_Phishing_URL_Dataset.csv` file
2. Run all cells
3. Download the trained model files
4. Place them in your local `models/` folder

**Expected time on T4 GPU: ~10-15 minutes**

## 1️⃣ Enable GPU

Go to: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**

In [ ]:
# Verify GPU is available
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

## 2️⃣ Install Dependencies

In [ ]:
!pip install -q tldextract validators

## 3️⃣ Upload Dataset

Click the upload button and select `PhiUSIIL_Phishing_URL_Dataset.csv`

In [ ]:
from google.colab import files
import os

print("📤 Upload your PhiUSIIL_Phishing_URL_Dataset.csv file:")
uploaded = files.upload()

# Get the uploaded filename
dataset_file = list(uploaded.keys())[0]
print(f"✅ Dataset uploaded: {dataset_file}")

## 4️⃣ Configuration & Utility Functions

In [ ]:
# Configuration
MAX_URL_LENGTH = 200
EMBEDDING_DIM = 128
LSTM_UNITS = 128
ATTENTION_UNITS = 64
DROPOUT_RATE = 0.3
BATCH_SIZE = 64
EPOCHS = 20
VALIDATION_SPLIT = 0.2

In [ ]:
# Utility Functions
import re
import tldextract
from urllib.parse import urlparse
import numpy as np

def extract_url_features(url):
    features = {}
    try:
        parsed_url = urlparse(url)
        ext = tldextract.extract(url)
        
        features['url_length'] = len(url)
        features['hostname_length'] = len(parsed_url.netloc)
        features['path_length'] = len(parsed_url.path)
        features['query_length'] = len(parsed_url.query)
        features['dot_count'] = url.count('.')
        features['dash_count'] = url.count('-')
        features['underscore_count'] = url.count('_')
        features['slash_count'] = url.count('/')
        features['question_count'] = url.count('?')
        features['equal_count'] = url.count('=')
        features['at_count'] = url.count('@')
        features['ampersand_count'] = url.count('&')
        features['exclamation_count'] = url.count('!')
        features['space_count'] = url.count(' ')
        features['tilde_count'] = url.count('~')
        features['comma_count'] = url.count(',')
        features['plus_count'] = url.count('+')
        features['asterisk_count'] = url.count('*')
        features['hash_count'] = url.count('#')
        features['dollar_count'] = url.count('$')
        features['percent_count'] = url.count('%')
        features['digit_count'] = sum(c.isdigit() for c in url)
        features['letter_count'] = sum(c.isalpha() for c in url)
        features['https'] = 1 if parsed_url.scheme == 'https' else 0
        features['http'] = 1 if parsed_url.scheme == 'http' else 0
        features['has_ip'] = 1 if re.match(r'\d+\.\d+\.\d+\.\d+', parsed_url.netloc) else 0
        features['has_port'] = 1 if ':' in parsed_url.netloc else 0
        features['subdomain_count'] = len(ext.subdomain.split('.')) if ext.subdomain else 0
        features['has_double_slash'] = 1 if '//' in parsed_url.path else 0
        features['has_at_symbol'] = 1 if '@' in url else 0
        features['tld_length'] = len(ext.suffix) if ext.suffix else 0
    except:
        for key in features.keys():
            features[key] = 0
    return features

def normalize_url(url):
    url = url.strip()
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url
    return url

## 5️⃣ Data Preprocessing

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

# Load dataset
print("Loading dataset...")
df = pd.read_csv(dataset_file)
urls = df['URL'].values
labels = df['label'].values.astype(int)

print(f"Loaded {len(urls)} URLs")
print(f"Phishing: {sum(labels)}, Legitimate: {len(labels) - sum(labels)}")

# Create and fit tokenizer
print("\nCreating tokenizer...")
normalized_urls = [normalize_url(url) for url in urls]
tokenizer = Tokenizer(num_words=10000, char_level=True)
tokenizer.fit_on_texts(normalized_urls)

# Transform URLs to sequences
print("Transforming URLs to sequences...")
sequences = tokenizer.texts_to_sequences(normalized_urls)
X_sequences = pad_sequences(sequences, maxlen=MAX_URL_LENGTH, padding='post', truncating='post')

# Extract features
print("Extracting features...")
features_list = []
for url in normalized_urls:
    features = extract_url_features(url)
    features_list.append(list(features.values()))
X_features = np.array(features_list)

# Scale features
scaler = StandardScaler()
X_features = scaler.fit_transform(X_features)

print(f"\nSequence shape: {X_sequences.shape}")
print(f"Features shape: {X_features.shape}")

# Split data
X_seq_train, X_seq_test, X_feat_train, X_feat_test, y_train, y_test = train_test_split(
    X_sequences, X_features, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training samples: {len(X_seq_train)}")
print(f"Test samples: {len(X_seq_test)}")

## 6️⃣ Build LSTM + Attention Model

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow import keras

# Custom Attention Layer
class AttentionLayer(layers.Layer):
    def __init__(self, units, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
        self.units = units
        
    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.u = self.add_weight(
            name='attention_context',
            shape=(self.units,),
            initializer='glorot_uniform',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)
        
    def call(self, x):
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        ait = tf.tensordot(uit, self.u, axes=1)
        attention_weights = tf.nn.softmax(ait, axis=1)
        attention_weights = tf.expand_dims(attention_weights, axis=-1)
        weighted_input = x * attention_weights
        output = tf.reduce_sum(weighted_input, axis=1)
        return output
    
    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config

# Build Model
print("Building model...")
sequence_input = layers.Input(shape=(MAX_URL_LENGTH,), name='sequence_input')
embedding = layers.Embedding(10000, EMBEDDING_DIM, input_length=MAX_URL_LENGTH)(sequence_input)

lstm_out = layers.Bidirectional(layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=DROPOUT_RATE))(embedding)
lstm_out = layers.Bidirectional(layers.LSTM(LSTM_UNITS // 2, return_sequences=True, dropout=DROPOUT_RATE))(lstm_out)
attention_out = AttentionLayer(ATTENTION_UNITS)(lstm_out)

lstm_dense = layers.Dense(64, activation='relu')(attention_out)
lstm_dense = layers.Dropout(DROPOUT_RATE)(lstm_dense)
lstm_dense = layers.Dense(32, activation='relu')(lstm_dense)

feature_input = layers.Input(shape=(X_features.shape[1],), name='feature_input')
feature_dense = layers.Dense(64, activation='relu')(feature_input)
feature_dense = layers.Dropout(DROPOUT_RATE)(feature_dense)
feature_dense = layers.Dense(32, activation='relu')(feature_dense)

concatenated = layers.Concatenate()([lstm_dense, feature_dense])
dense = layers.Dense(64, activation='relu')(concatenated)
dense = layers.Dropout(DROPOUT_RATE)(dense)
dense = layers.Dense(32, activation='relu')(dense)
dense = layers.Dropout(DROPOUT_RATE)(dense)
output = layers.Dense(1, activation='sigmoid')(dense)

model = Model(inputs=[sequence_input, feature_input], outputs=output, name='phishing_detector_lstm_attention')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall(), keras.metrics.AUC()]
)

model.summary()

## 7️⃣ Train Model (This is the main step - will take ~10-15 minutes on T4 GPU)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('phishing_detector_lstm_attention.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1, min_lr=1e-7)
]

print("\n🚀 Starting training on GPU...\n")
history = model.fit(
    [X_seq_train, X_feat_train],
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training completed!")

## 8️⃣ Evaluate Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

# Predictions
y_pred_proba = model.predict([X_seq_test, X_feat_test])
y_pred = (y_pred_proba > 0.5).astype(int)

# Metrics
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300)
plt.show()

print(f"\n✅ Test Accuracy: {np.mean(y_pred == y_test):.4f}")
print(f"✅ Test AUC: {roc_auc:.4f}")

## 9️⃣ Save Preprocessor Files

In [ ]:
# Save tokenizer
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print("✅ Tokenizer saved: tokenizer.pkl")

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved: scaler.pkl")

print("\n📦 All files ready for download!")

## 🔟 Download Model Files

Download these 3 files and place them in your local `models/` folder:

In [ ]:
from google.colab import files

print("📥 Downloading files...\n")

# Download model
files.download('phishing_detector_lstm_attention.h5')
print("✅ Downloaded: phishing_detector_lstm_attention.h5")

# Download tokenizer
files.download('tokenizer.pkl')
print("✅ Downloaded: tokenizer.pkl")

# Download scaler
files.download('scaler.pkl')
print("✅ Downloaded: scaler.pkl")

print("\n🎉 All done! Place these files in your local models/ folder and run:")
print("   python backend/app.py")
print("   streamlit run frontend/streamlit_app.py")

## 📝 Next Steps

1. Download all 3 files from Colab
2. Place them in your project's `models/` folder:
   - `phishing_detector_lstm_attention.h5`
   - `tokenizer.pkl`
   - `scaler.pkl`
3. Start the backend: `python backend/app.py`
4. Start the frontend: `streamlit run frontend/streamlit_app.py`
5. Test your phishing detector! 🚀